# Practice 62 — pandas + NumPy

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns

---

## Level 1 — Staff payroll

`staff` records salary and bonus data for eleven employees across three departments and two seniority levels.

**New concept: named aggregations**

Pass keyword arguments to `.agg()` in the form `output_name=(source_column, function)`. The keyword becomes the column name in the result — flat output, no renaming needed:

```python
df.groupby('dept').agg(
    headcount=('emp_id', 'count'),
    avg_salary=('salary', 'mean'),
    total_bonus=('bonus', 'sum'),
)
```

`pd.NamedAgg(column=..., aggfunc=...)` is the explicit form — same result, but useful when `aggfunc` is a custom lambda:

```python
df.groupby('dept').agg(
    headcount=pd.NamedAgg(column='emp_id', aggfunc='count'),
)
```

1. Group by `dept`. Use named agg to compute `headcount` (count of `emp_id`), `avg_salary` (mean of `salary`), and `bonus_total` (sum of `bonus`).
2. Group by `dept` and `level`. Use a lambda as the `aggfunc` to compute `salary_spread` (max minus min of `salary`). Which (`dept`, `level`) pair has the widest spread?
3. From your step-1 summary, use `np.argmax` on `avg_salary` to identify the top-paying department. Confirm with `.idxmax()` — do they agree?

In [11]:
staff = pd.DataFrame({
    'emp_id': ['E01','E02','E03','E04','E05','E06','E07','E08','E09','E10','E11'],
    'dept':   ['Eng','Eng','Eng','Eng','Mkt','Mkt','Mkt','Sales','Sales','Sales','Sales'],
    'level':  ['Junior','Junior','Senior','Senior','Junior','Senior','Senior','Junior','Junior','Senior','Senior'],
    'salary': [72, 78, 105, 98, 65, 90, 95, 60, 63, 88, 115],
    'bonus':  [5, 6, 12, 10, 6, 9, 11, 4, 5, 8, 14],
})

# Your code here

s = staff.groupby('dept').agg(
    headcount = ('emp_id','count'),
    avg_salary = ('salary','mean'),
    total_bonus = ('bonus','sum')
)
print(s)
p = staff.groupby(['dept','level']).agg(salary_range=pd.NamedAgg(column= 'salary', aggfunc=lambda x: x.max() - x.min()))
print(p)
print(p['salary_range'].idxmax(),'has the widest range')

print('use np.argmax:', s.index[np.argmax(s['avg_salary'])])
print('use idxmax: ', s['avg_salary'].idxmax())

       headcount  avg_salary  total_bonus
dept                                     
Eng            4   88.250000           33
Mkt            3   83.333333           26
Sales          4   81.500000           31
              salary_range
dept  level               
Eng   Junior             6
      Senior             7
Mkt   Junior             0
      Senior             5
Sales Junior             3
      Senior            27
('Sales', 'Senior') has the widest range
use np.argmax: Eng
use idxmax:  Eng


---

## Level 2 — Tips

Use `sns.load_dataset('tips')`.

1. Add `tip_pct = tip / total_bill`. Group by `day` and `smoker`. Use named agg to compute `n` (count of `tip`), `avg_tip_pct` (mean of `tip_pct`), and `high_bill` (max of `total_bill`).
2. What are the 25th and 75th percentiles of `avg_tip_pct` across all eight groups? Which (day, smoker) combo has the highest average tip percent?
3. Across the eight groups, does higher average tip percent tend to come with higher maximum bills? Check with `np.corrcoef`.

In [18]:
tips = sns.load_dataset('tips')

# Your code here

tips['tip_pct'] = tips['tip']/tips['total_bill']

gg = tips.groupby(['day','smoker'], observed=True).agg(
    n = ('tip','count'),
    avg_tip_pct = ('tip_pct','mean'),
    high_bill = ('total_bill','max')
)

print(gg)


print(np.percentile(gg['avg_tip_pct'], q=[25,75]))

print(gg['avg_tip_pct'].idxmax())

c = np.corrcoef(gg['avg_tip_pct'], gg['high_bill'])[0,1]
print(c,'weakly correlated')

              n  avg_tip_pct  high_bill
day  smoker                            
Thur Yes     17     0.163863      43.11
     No      45     0.160298      41.19
Fri  Yes     15     0.174783      40.17
     No       4     0.151650      22.75
Sat  Yes     42     0.147906      50.81
     No      45     0.158048      48.33
Sun  Yes     19     0.187250      45.35
     No      57     0.160113      48.17
[0.15644835 0.16659322]
('Sun', 'Yes')
0.12720530318282697 weakly correlated


---

## Level 3 — Diamonds

Use `sns.load_dataset('diamonds')`. Build a **cut-level summary** (group by `cut`, `observed=True`) using named aggregations. Include at least: median price, mean price, std of price, mean carat, and row count. Then answer the questions below — no sub-steps, just write the code that gets each answer.

- Which cut has the highest median price? Which has the most price variability (std)?
- Rank cuts from most to least expensive by mean price. Use `np.argsort`.
- Across cuts, is mean carat correlated with mean price? Check with `np.corrcoef`.
- Build a second summary grouped by `color` (`observed=True`) using `pd.NamedAgg` syntax explicitly (not the tuple shorthand). Compute `price_std` and `count`. Which color shows the most price variability?

In [32]:
diamonds = sns.load_dataset('diamonds')

# Your code here
cl = diamonds.groupby('cut', observed=True).agg(
    median_price = ('price','median'),
    mean_price = ('price','mean'),
    std_price = ('price','std'),
    mean_carat = ('carat','mean'),
    row_count = ('carat','count')
)
print(cl)

print(cl['median_price'].idxmax(),'has the highest median price.')
print(cl['std_price'].idxmax(),'has the most price variability.')
print(cl.iloc[np.argsort(-cl['mean_price'])].index)
c = np.corrcoef(cl['mean_carat'],cl['mean_price'])[0,1]
print(f'correltion is {c:.2f}, positively correlated')

ss = diamonds.groupby('color', observed=True).agg(
    price_std = pd.NamedAgg(column='price', aggfunc = 'std'),
    count = pd.NamedAgg(column='price',aggfunc='count')

)
print(ss)

print(ss['price_std'].idxmax(),'shows the most price variablity ')

           median_price   mean_price    std_price  mean_carat  row_count
cut                                                                     
Ideal            1810.0  3457.541970  3808.401172    0.702837      21551
Premium          3185.0  4584.257704  4349.204961    0.891955      13791
Very Good        2648.0  3981.759891  3935.862161    0.806381      12082
Good             3050.5  3928.864452  3681.589584    0.849185       4906
Fair             3282.0  4358.757764  3560.386612    1.046137       1610
Fair has the highest median price.
Premium has the most price variability.
CategoricalIndex(['Premium', 'Fair', 'Very Good', 'Good', 'Ideal'], categories=['Ideal', 'Premium', 'Very Good', 'Good', 'Fair'], ordered=False, dtype='category', name='cut')
correltion is 0.79, positively correlated
         price_std  count
color                    
D      3356.590935   6775
E      3344.158685   9797
F      3784.992007   9542
G      4051.102846  11292
H      4215.944171   8304
I      4722.387